In [ ]:
import pandas as pd
import numpy as np
from google.cloud import bigquery
import plotly.express as px
from utils import read_config

config = read_config.load_config()

### Connect to BQ client

In [ ]:
namespaces = config['bigquery']['namespaces']
OUTPUT_NAMESPACE = namespaces['output']
OMOP_NAMESPACE = namespaces['omop']
EHR_NAMESPACE = namespaces['ehr']
MAPPING_NAMESPACE = namespaces['mappings']
COHORT_TABLE = config['analysis_configs']['COHORT_TABLE']
COHORT_LABEL_PREFIX = config['analysis_configs']['COHORT_LABEL_PREFIX']
ANALYSIS_END_DATE = config['omop_table_configs']['ANALYSIS_END_DATE']

In [ ]:
client = bigquery.Client()

### Phecode Analysis

#### Heatmap - Phecode Category Level

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.cm as cm
import math

Total 54 hpos in V8 deid dataset. 49 have mapped Phecodes.



In [ ]:
phecode_category_query = f"""
SELECT ms.src_hpo_id, ms.site_id, phe.PhecodeCategory,
count(distinct ls.patient_num_rid) AS patient_cnt, avg(ls.score) as phecode_category_level_avg_loyalty_score
FROM `{OUTPUT_NAMESPACE}.{COHORT_TABLE}` AS ls
JOIN `{OMOP_NAMESPACE}.condition_occurrence` AS co
ON ls.patient_num_rid = co.person_id
JOIN `{OMOP_NAMESPACE}.condition_occurrence_ext` AS mc
ON co.condition_occurrence_id = mc.condition_occurrence_id
JOIN `{OMOP_NAMESPACE}.concept` AS c
ON c.concept_id = co.condition_source_concept_id
JOIN `{OUTPUT_NAMESPACE}.phecode_mapx_filtered` AS phe
ON phe.ICD = c.concept_code
JOIN `{MAPPING_NAMESPACE}.site_maskings` sm
ON ls.src_id = sm.src_id
JOIN `{OUTPUT_NAMESPACE}.mask_site_id` AS ms
ON sm.hpo_id = ms.src_hpo_id
WHERE ls.cohort_name = '{COHORT_LABEL_PREFIX}_{ANALYSIS_END_DATE}_1years'
AND c.vocabulary_id in ('ICD9CM', 'ICD10CM')
AND mc.src_id LIKE '%EHR site%'
GROUP BY ms.src_hpo_id, ms.site_id, phe.PhecodeCategory
"""

In [ ]:
hpo_phecode_category = client.query(phecode_category_query).to_dataframe()

In [ ]:
hpo_phecode_category['site_id'].nunique()

In [ ]:
patient_count_query = f"""
SELECT sm.hpo_id AS src_hpo_id, count(distinct patient_num_rid) as pt_total_cnt
FROM `{OUTPUT_NAMESPACE}.{COHORT_TABLE}` ls
JOIN `{MAPPING_NAMESPACE}.site_maskings` sm
ON ls.src_id = sm.src_id
WHERE cohort_name = '{COHORT_LABEL_PREFIX}_{ANALYSIS_END_DATE}_1years'
GROUP BY sm.hpo_id
"""
patient_count_hpo_df = client.query(patient_count_query).to_dataframe()

In [ ]:
hpo_phecode_category = hpo_phecode_category.merge(patient_count_hpo_df, on='src_hpo_id', how='left')
hpo_phecode_category['perc'] = hpo_phecode_category['patient_cnt'] / hpo_phecode_category['pt_total_cnt']

In [ ]:
hpo_phecode_category_filter = hpo_phecode_category[hpo_phecode_category['pt_total_cnt'] > 100]

Exclude 2 sites with participant count less or equal than 100

In [ ]:
hpo_phecode_category[hpo_phecode_category['pt_total_cnt'] <= 100]['site_id'].nunique()

Exclude site 14 and site 26 due to less of ICD codes in source code

In [ ]:
hpo_phecode_category_filter = hpo_phecode_category_filter[~hpo_phecode_category_filter['site_id'].isin(['site_14', 'site_26'])]

In [ ]:
hpo_phecode_category_pivot = hpo_phecode_category_filter.pivot(index=['PhecodeCategory'], columns='site_id', values='perc').fillna(0).astype(float)

In [ ]:
loyalty_score_per_site_query = f"""
SELECT sm.hpo_id AS src_hpo_id, ms.site_id, avg(score) as avg_hpo_loyalty_score
FROM `{OUTPUT_NAMESPACE}.{COHORT_TABLE}` AS ls
JOIN `{MAPPING_NAMESPACE}.site_maskings` sm
ON ls.src_id = sm.src_id
JOIN `{OUTPUT_NAMESPACE}.mask_site_id` AS ms
ON sm.hpo_id = ms.src_hpo_id
WHERE ls.cohort_name = '{COHORT_LABEL_PREFIX}_{ANALYSIS_END_DATE}_1years'
GROUP BY sm.hpo_id, site_id
"""
loyalty_score_per_site_df = client.query(loyalty_score_per_site_query).to_dataframe()

In [ ]:
hpo_phecode_category_pivot.columns

In [ ]:
hpo_phecode_category_pivot['mean'] = hpo_phecode_category_pivot.mean(axis=1)

In [ ]:
hpo_phecode_category_pivot[['mean']].sort_values(by="mean", ascending=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# --- Step 1: Extract site loyalty as a Series ---
loyalty_series = (
    loyalty_score_per_site_df
    .set_index("site_id")["avg_hpo_loyalty_score"]
    .sort_index()
)

# --- Step 2: Align both dataframes to the same site order ---
common_sites = hpo_phecode_category_pivot.columns.intersection(loyalty_series.index)
hpo_phecode_category_pivot_aligned = hpo_phecode_category_pivot[common_sites]
loyalty_series_aligned = loyalty_series.reindex(common_sites)

# --- Step 3: Plot ---
fig = plt.figure(figsize=(15, 6), constrained_layout=True)  # Changed here
gs = fig.add_gridspec(2, 1, height_ratios=[4, 1], hspace=0.05)

# Top: Heatmap
ax0 = fig.add_subplot(gs[0])
sns.heatmap(hpo_phecode_category_pivot_aligned, ax=ax0, cmap="Blues", cbar=True)
ax0.set_title("Phecode Category Heatmap by Site")
ax0.set_xlabel("")

# Bottom: Loyalty Score line
ax1 = fig.add_subplot(gs[1], sharex=ax0)

# Heatmap columns are centered at 0.5, 1.5, 2.5, ...
x_positions = [i + 0.5 for i in range(len(loyalty_series_aligned))]

ax1.bar(
    x_positions,
    loyalty_series_aligned.values,
    width=0.8,
    color="darkred",
    alpha=0.8
)

ax1.set_ylabel("Loyalty Score", fontsize=11)
ax1.set_ylim(0, 1)
ax1.set_xlabel("Site", fontsize=11)
ax1.set_xticks(x_positions)
ax1.set_xticklabels(loyalty_series_aligned.index, rotation=45, ha="right")

# Hide top x tick labels
plt.setp(ax0.get_xticklabels(), visible=False)

fig.savefig(
    'figures/phecode_distribution.png',
    dpi=300,
    bbox_inches='tight',
    facecolor='white',
)
plt.show()  # Save the named figure before displaying it

In [ ]:
plt.figure(figsize=(15, 4))
sns.heatmap(hpo_phecode_category_pivot, annot=False)
plt.show()

##### Manhattan Plot - Phecode Level

In [ ]:
phecode_level_query = f"""
SELECT sm.hpo_id AS src_hpo_id, ms.site_id, phe.PhecodeCategory, phe.Phecode, phe.PhecodeString,
count(distinct ls.patient_num_rid) AS patient_cnt, avg(ls.score) as phecode_level_avg_loyalty_score
FROM `{OUTPUT_NAMESPACE}.{COHORT_TABLE}` AS ls
JOIN `{OMOP_NAMESPACE}.condition_occurrence` AS co
ON ls.patient_num_rid = co.person_id
JOIN `{OMOP_NAMESPACE}.concept` AS c
ON c.concept_id = co.condition_source_concept_id
JOIN `{OUTPUT_NAMESPACE}.phecode_mapx_filtered` AS phe
ON phe.ICD = c.concept_code
JOIN `{MAPPING_NAMESPACE}.site_maskings` sm
ON ls.src_id = sm.src_id
JOIN `{OUTPUT_NAMESPACE}.mask_site_id` AS ms
ON sm.hpo_id = ms.src_hpo_id
WHERE ls.cohort_name = '{COHORT_LABEL_PREFIX}_{ANALYSIS_END_DATE}_1years'
AND c.vocabulary_id in ('ICD9CM', 'ICD10CM')
GROUP BY sm.hpo_id, ms.site_id, phe.PhecodeCategory, phe.Phecode, phe.PhecodeString
"""
phecode_df = client.query(phecode_level_query).to_dataframe()

In [ ]:
patient_count_hpo_df

In [ ]:
phecode_df = phecode_df.merge(patient_count_hpo_df, on='src_hpo_id', how='left')

In [ ]:
phecode_df

In [ ]:
phecode_df.columns

In [ ]:

phecode_df['perc'] = phecode_df['patient_cnt'] / phecode_df['pt_total_cnt']

hpo_phecode = phecode_df

In [ ]:
hpo_phecode['mean_perc'] = hpo_phecode.groupby('Phecode')['perc'].transform('mean')

In [ ]:
hpo_phecode_filter = hpo_phecode[hpo_phecode['pt_total_cnt'] >= 100]

In [ ]:
pheinfo = pd.read_csv('phecodeX_info.csv', encoding='latin1')
hpo_phecode_filter = hpo_phecode_filter.merge(pheinfo, left_on='Phecode', right_on='phecode', how='left')
hpo_phecode_pivot = hpo_phecode_filter.pivot(index=['PhecodeCategory', 'Phecode', 'phecode_string'], columns='site_id', values='perc').fillna(0).astype(float)

In [ ]:
hpo_phecode_pivot['mean'] = hpo_phecode_pivot.mean(axis=1)
hpo_phecode_pivot['std'] = hpo_phecode_pivot.std(axis=1)

In [ ]:
hpo_phecode_pivot.reset_index(inplace=True)

In [ ]:
hpo_phecode_filter = pd.merge(hpo_phecode_filter, loyalty_score_per_site_df, on='src_hpo_id',  how='left')

In [ ]:
hpo_phecode_filter = hpo_phecode_filter.drop('site_id_y', axis=1).rename(columns={'site_id_x':'site_id'})

In [ ]:
hpo_phecode_filter.groupby('site_id').max()['avg_hpo_loyalty_score']

In [ ]:
def plot_manhattan_style(hpo_name, hpo_phecode):
    test_df = hpo_phecode[hpo_phecode['site_id'] == hpo_name]

    test_df = test_df.reset_index(drop=True)

    test_df.PhecodeCategory = test_df.PhecodeCategory.astype(str)
    categories = sorted(test_df['PhecodeCategory'].unique())
    pt_cnt = test_df['pt_total_cnt'].values[0]
    loyalty_score = test_df['avg_hpo_loyalty_score'].values[0]
    
    if pt_cnt < 20:
        return None
    else:
        # Assign an x-position with equal spacing per category
        cat_to_xstart = {cat: i * 10 for i, cat in enumerate(categories)}  # spacing = 10
        test_df['x'] = test_df.apply(
            lambda row: cat_to_xstart[row['PhecodeCategory']] + np.random.uniform(0, 10),
            axis=1
        )

        # Generate a color for each category using a colormap
        num_colors = len(categories)
        colors = cm.get_cmap('tab20', num_colors)
        cat_to_color = {cat: colors(i) for i, cat in enumerate(categories)}

        # Map colors to each row
        test_df['color'] = test_df['PhecodeCategory'].map(cat_to_color)

        # Plot
        plt.figure(figsize=(16, 8))
        plt.scatter(test_df['x'], test_df['perc'], c=test_df['color'], alpha=0.7)
        plt.scatter(test_df['x'], test_df['mean_perc'], c='grey', marker='x', alpha=0.3, s=80, label='Mean Across HPOs')


        # Set category tick positions and labels
        tick_positions = [v + 5 for v in cat_to_xstart.values()]  # middle of each block
        plt.xticks(tick_positions, categories, rotation=90)
        plt.ylim(0, 1)

        # Labels and title
        plt.xlabel('Phecode Category')
        plt.ylabel('Patient Percentage')
        plt.title(f'Manhattan-style Plot of Patient Counts by Phecode Category for {hpo_name} with {int(pt_cnt)} Participants, Avg Loyalty Score: {loyalty_score:.2f}')

        plt.tight_layout()
        plt.savefig(
            f'figures/{hpo_name}_distribution.png',
            dpi=300,
            bbox_inches='tight',
            facecolor='white',
        )
        plt.show()

In [ ]:
for hpo_name in hpo_phecode_filter['site_id'].unique():
    if hpo_name in ['site_26']:
        plot_manhattan_style(hpo_name, hpo_phecode_filter)


## Reviewer-response analyses: representativeness, standardization, and mapping diagnostics

The analyses below avoid selecting a single illustrative site after inspecting the results. Low-loyalty sites are defined *a priori* as sites in the lowest quartile of mean 5-year loyalty score. Site 17 is compared with every site in that group, while Sites 40 and 46 are included in a focused diagnostic table.

Crude Phecode prevalence alone cannot establish a mapping error. Therefore, the notebook also calculates (1) age-standardized Phecode-category prevalence and (2) the proportion of eligible ICD-9-CM/ICD-10-CM condition records that match the Phecode map. These analyses can identify patterns consistent with mapping limitations, but they do not prove their cause.

### ICD-to-Phecode mapping diagnostics

In [ ]:
mapping_diagnostics_query = f"""
WITH cohort AS (
  SELECT DISTINCT
    ls.patient_num_rid AS person_id,
    ms.src_hpo_id,
    ms.site_id
  FROM `{OUTPUT_NAMESPACE}.{COHORT_TABLE}` AS ls
  JOIN `{MAPPING_NAMESPACE}.site_maskings` sm
    ON ls.src_id = sm.src_id
  JOIN `{OUTPUT_NAMESPACE}.mask_site_id` AS ms
    ON sm.hpo_id = ms.src_hpo_id
  WHERE ls.cohort_name = '{COHORT_LABEL_PREFIX}_{ANALYSIS_END_DATE}_5years'
),
mapped_icd AS (
  SELECT DISTINCT ICD
  FROM `{OUTPUT_NAMESPACE}.phecode_mapx_filtered`
),
eligible_conditions AS (
  SELECT
    c.person_id,
    c.site_id,
    co.condition_occurrence_id,
    concept.concept_code,
    mapped_icd.ICD IS NOT NULL AS is_mapped
  FROM cohort AS c
  JOIN `{OMOP_NAMESPACE}.condition_occurrence` AS co
    ON c.person_id = co.person_id
  JOIN `{OMOP_NAMESPACE}.concept` AS concept
    ON concept.concept_id = co.condition_source_concept_id
  LEFT JOIN mapped_icd
    ON mapped_icd.ICD = concept.concept_code
  WHERE concept.vocabulary_id IN ('ICD9CM', 'ICD10CM')
)
SELECT
  c.site_id,
  COUNT(DISTINCT c.person_id) AS cohort_patients,
  COUNT(DISTINCT ec.person_id) AS patients_with_eligible_icd,
  COUNT(DISTINCT IF(ec.is_mapped, ec.person_id, NULL)) AS patients_with_mapped_phecode,
  COUNT(DISTINCT ec.condition_occurrence_id) AS eligible_icd_records,
  COUNT(DISTINCT IF(ec.is_mapped, ec.condition_occurrence_id, NULL)) AS mapped_icd_records
FROM cohort AS c
LEFT JOIN eligible_conditions AS ec
  ON c.person_id = ec.person_id AND c.site_id = ec.site_id
GROUP BY c.site_id
ORDER BY c.site_id
"""

mapping_diagnostics = client.query(mapping_diagnostics_query).to_dataframe()
mapping_diagnostics['eligible_icd_patient_coverage'] = (
    mapping_diagnostics['patients_with_eligible_icd'] / mapping_diagnostics['cohort_patients']
)
mapping_diagnostics['mapped_phecode_patient_coverage'] = (
    mapping_diagnostics['patients_with_mapped_phecode'] / mapping_diagnostics['cohort_patients']
)
mapping_diagnostics['icd_record_mapping_rate'] = (
    mapping_diagnostics['mapped_icd_records'] / mapping_diagnostics['eligible_icd_records']
)
mapping_diagnostics.head()

In [ ]:
mapping_diagnostics

### Direct age standardization of Phecode-category prevalence

The pooled age distribution across all included HPOs is used as the standard population. Broad age groups limit sparse cells. Sites without observations in every standard age stratum are flagged through `age_strata_covered`; their standardized estimates should be interpreted cautiously.

In [ ]:
age_standardization_query = f"""
WITH cohort AS (
  SELECT DISTINCT
    ls.patient_num AS person_id,
    ms.site_id,
    CASE
      WHEN ls.age < 18 THEN '0-17'
      WHEN ls.age < 35 THEN '18-34'
      WHEN ls.age < 50 THEN '35-49'
      WHEN ls.age < 65 THEN '50-64'
      WHEN ls.age < 75 THEN '65-74'
      ELSE '75+'
    END AS age_group
  FROM `{OUTPUT_NAMESPACE}.{COHORT_TABLE}` AS ls
  JOIN `{EHR_NAMESPACE}._mapping_person` AS mp
    ON ls.patient_num = mp.person_id
  JOIN `{OUTPUT_NAMESPACE}.mask_site_id` AS ms
    ON mp.src_hpo_id = ms.src_hpo_id
  WHERE ls.cohort_name = '{COHORT_LABEL_PREFIX}_{ANALYSIS_END_DATE}_5years'
    AND ls.age IS NOT NULL
),
mapped_categories AS (
  SELECT DISTINCT
    co.person_id,
    phe.PhecodeCategory
  FROM `{EHR_NAMESPACE}.unioned_ehr_condition_occurrence` AS co
  JOIN `{EHR_NAMESPACE}.concept` AS concept
    ON concept.concept_id = co.condition_source_concept_id
  JOIN `{OUTPUT_NAMESPACE}.phecode_mapx_filtered` AS phe
    ON phe.ICD = concept.concept_code
  WHERE concept.vocabulary_id IN ('ICD9CM', 'ICD10CM')
),
categories AS (
  SELECT DISTINCT PhecodeCategory
  FROM `{OUTPUT_NAMESPACE}.phecode_mapx_filtered`
)
SELECT
  c.site_id,
  c.age_group,
  categories.PhecodeCategory,
  COUNT(DISTINCT c.person_id) AS stratum_patients,
  COUNT(DISTINCT IF(mc.person_id IS NOT NULL, c.person_id, NULL)) AS phecode_patients
FROM cohort AS c
CROSS JOIN categories
LEFT JOIN mapped_categories AS mc
  ON c.person_id = mc.person_id
  AND categories.PhecodeCategory = mc.PhecodeCategory
GROUP BY c.site_id, c.age_group, categories.PhecodeCategory
ORDER BY c.site_id, c.age_group, categories.PhecodeCategory
"""

age_category_counts = client.query(age_standardization_query).to_dataframe()
age_category_counts['age_specific_prevalence'] = (
    age_category_counts['phecode_patients'] / age_category_counts['stratum_patients']
)

In [ ]:
# Denominators repeat across Phecode categories; retain one site-age row for the standard population.
age_denominators = (
    age_category_counts[['site_id', 'age_group', 'stratum_patients']]
    .drop_duplicates()
)
standard_age_weights = (
    age_denominators.groupby('age_group', as_index=False)['stratum_patients'].sum()
    .rename(columns={'stratum_patients': 'standard_patients'})
)
standard_age_weights['standard_weight'] = (
    standard_age_weights['standard_patients'] / standard_age_weights['standard_patients'].sum()
)

age_standardized = age_category_counts.merge(
    standard_age_weights[['age_group', 'standard_weight']],
    on='age_group',
    how='left',
)
age_standardized['weighted_prevalence'] = (
    age_standardized['age_specific_prevalence'] * age_standardized['standard_weight']
)
age_standardized_category = (
    age_standardized.groupby(['site_id', 'PhecodeCategory'], as_index=False)
    .agg(
        age_standardized_prevalence=('weighted_prevalence', 'sum'),
        covered_standard_weight=('standard_weight', 'sum'),
        age_strata_covered=('age_group', 'nunique'),
    )
)
age_standardized_category['age_standardized_prevalence'] /= (
    age_standardized_category['covered_standard_weight']
)
age_standardized_category.head()

### Is Site 17 representative of other low-loyalty sites?

For each low-loyalty site, its age-standardized category profile is compared with the median profile of the *other* low-loyalty sites. Pearson profile correlation measures similarity of the pattern across categories, while root mean squared error (RMSE) measures absolute deviation. This leave-one-site-out comparison prevents the evaluated site from contributing to its own reference profile.

In [ ]:
site_loyalty = (
    loyalty_score_per_site_df[['site_id', 'avg_hpo_loyalty_score']]
    .drop_duplicates('site_id')
)
low_loyalty_cutoff = site_loyalty['avg_hpo_loyalty_score'].quantile(0.25)
low_loyalty_sites = site_loyalty.loc[
    site_loyalty['avg_hpo_loyalty_score'] <= low_loyalty_cutoff, 'site_id'
].tolist()

standardized_profile = age_standardized_category.pivot(
    index='site_id',
    columns='PhecodeCategory',
    values='age_standardized_prevalence',
)

def compare_with_reference(site_id, reference_sites):
    reference_sites = [site for site in reference_sites if site != site_id]
    site_profile = standardized_profile.loc[site_id]
    reference_profile = standardized_profile.loc[reference_sites].median(axis=0)
    complete = pd.concat([site_profile, reference_profile], axis=1).dropna()
    complete.columns = ['site', 'reference']
    return pd.Series({
        'profile_correlation': complete['site'].corr(complete['reference']),
        'profile_rmse': np.sqrt(np.mean((complete['site'] - complete['reference']) ** 2)),
        'categories_compared': len(complete),
    })

def compare_with_other_low_loyalty_sites(site_id):
    return compare_with_reference(site_id, low_loyalty_sites)

representativeness = pd.DataFrame([
    {'site_id': site, **compare_with_other_low_loyalty_sites(site).to_dict()}
    for site in low_loyalty_sites
]).merge(site_loyalty, on='site_id', how='left')
representativeness['rmse_percentile_among_low_loyalty_sites'] = (
    representativeness['profile_rmse'].rank(pct=True)
)

all_site_profile_comparison = pd.DataFrame([
    {'site_id': site, **compare_with_reference(site, standardized_profile.index).to_dict()}
    for site in standardized_profile.index
]).rename(columns={
    'profile_correlation': 'overall_profile_correlation',
    'profile_rmse': 'overall_profile_rmse',
    'categories_compared': 'overall_categories_compared',
})
all_site_profile_comparison['overall_rmse_percentile'] = (
    all_site_profile_comparison['overall_profile_rmse'].rank(pct=True)
)

representativeness.sort_values('profile_rmse', ascending=False)

In [ ]:
focus_sites = ['site_17', 'site_40', 'site_46']
site_phecode_counts = (
    hpo_phecode_filter.groupby('site_id', as_index=False)['Phecode']
    .nunique()
    .rename(columns={'Phecode': 'distinct_observed_phecodes'})
)

focus_site_diagnostics = (
    site_loyalty.merge(mapping_diagnostics, on='site_id', how='left')
    .merge(site_phecode_counts, on='site_id', how='left')
    .merge(all_site_profile_comparison, on='site_id', how='left')
    .merge(
        representativeness[['site_id', 'profile_correlation', 'profile_rmse',
                              'rmse_percentile_among_low_loyalty_sites']],
        on='site_id',
        how='left',
    )
)
focus_site_diagnostics['loyalty_percentile'] = (
    focus_site_diagnostics['avg_hpo_loyalty_score'].rank(pct=True)
)
focus_site_diagnostics.loc[
    focus_site_diagnostics['site_id'].isin(focus_sites),
    [
        'site_id', 'cohort_patients', 'avg_hpo_loyalty_score', 'loyalty_percentile',
        'eligible_icd_patient_coverage', 'mapped_phecode_patient_coverage',
        'icd_record_mapping_rate', 'distinct_observed_phecodes',
        'overall_profile_correlation', 'overall_profile_rmse',
        'overall_rmse_percentile',
        'profile_correlation', 'profile_rmse',
        'rmse_percentile_among_low_loyalty_sites',
    ],
].sort_values('site_id')

In [ ]:
plot_sites = [site for site in focus_sites if site in standardized_profile.index]
low_loyalty_reference = standardized_profile.loc[low_loyalty_sites].median(axis=0)
profile_plot_data = (
    standardized_profile.loc[plot_sites]
    .T.assign(low_loyalty_site_median=low_loyalty_reference)
    .reset_index()
    .melt(
        id_vars='PhecodeCategory',
        var_name='site_id',
        value_name='age_standardized_prevalence',
    )
)

plt.figure(figsize=(12, 6))
sns.lineplot(
    data=profile_plot_data,
    x='PhecodeCategory',
    y='age_standardized_prevalence',
    hue='site_id',
    marker='o',
)
plt.xticks(rotation=90)
plt.xlabel('Phecode category')
plt.ylabel('Age-standardized prevalence')
plt.title('Age-standardized Phecode profiles for selected sites')
plt.tight_layout()
plt.show()

### Interpretation guide

- Site 17 is reasonably representative if its profile correlation is similar to those of the other low-loyalty sites and its RMSE is not near the top of their distribution. If its RMSE percentile is high or its correlation is unusually low, describe it as an anomalous example rather than a representative one.
- A low crude Phecode prevalence that becomes less unusual after age standardization is consistent with age composition contributing to the original difference.
- Low mapped-Phecode patient coverage accompanied by a typical ICD-record mapping rate is more consistent with differences in diagnosis burden, healthcare utilization, or case mix than with failure of the mapping dictionary.
- An unusually low ICD-record mapping rate indicates that a larger share of the site's eligible source codes is absent from the mapping table. This is consistent with a coding or mapping limitation but does not by itself establish an upstream data-quality error.
- Residual differences after age standardization may still reflect sex, race/ethnicity, morbidity, utilization, coding practices, or other case-mix factors. Avoid causal language unless these are addressed.